### Importação das bibliotecas e carregamento dos dados pré-processados

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd

from src.preprocessing import load_processed_data
from src.evaluation import evaluate_model, save_results

# Carregando os conjuntos já pré-processados (One-Hot Encoding + SMOTE no treino),
# gerados pelo notebook 01 e compartilhados entre os modelos da equipe
X_train_res, X_val, X_test, y_train_res, y_val, y_test, preprocessador = load_processed_data('../data/processed')

print(f"Dimensão de X_train_res: {X_train_res.shape}")
print(f"Dimensão de X_val: {X_val.shape}")
print(f"Dimensão de X_test: {X_test.shape}")


### Treinamento do modelo base

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Modelo com hiperparâmetros padrão, usado como referência antes da tunagem
modelo_base = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
modelo_base.fit(X_train_res, y_train_res)


### Avaliação do modelo base no conjunto de validação

In [ ]:
metricas_base = evaluate_model(modelo_base, X_val, y_val, model_name='Random Forest (base)')


### Tunagem de hiperparâmetros

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Busca aleatória sobre o número de árvores, profundidade máxima e tamanho
# mínimo das folhas, usando AUC-ROC como métrica de seleção
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [None, 10, 20, 30],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
}

busca = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=10,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=-1,
)

busca.fit(X_train_res, y_train_res)

print("Melhores hiperparâmetros:", busca.best_params_)
print(f"Melhor AUC-ROC (CV): {busca.best_score_:.4f}")

modelo_otimizado = busca.best_estimator_


### Avaliação do modelo otimizado no conjunto de validação

In [ ]:
metricas_val = evaluate_model(modelo_otimizado, X_val, y_val, model_name='Random Forest')


### Importância das variáveis

In [ ]:
# As features mais importantes ajudam a entender quais atributos
# (recurso, gerente, papel) mais influenciam a decisão de acesso
importancias = pd.Series(
    modelo_otimizado.feature_importances_,
    index=preprocessador.get_feature_names_out(),
).sort_values(ascending=False)

importancias.head(20)


### Avaliação final no conjunto de teste interno

In [ ]:
metricas_teste = evaluate_model(modelo_otimizado, X_test, y_test, model_name='Random Forest')

# Salvando as métricas para a comparação entre modelos na Fase 3
save_results(metricas_teste, output_path='../results/resultados_modelos.csv')


### Salvando o modelo treinado

In [ ]:
import joblib
from pathlib import Path

# Modelo salvo para reuso, sem precisar repetir o treinamento e a tunagem
Path('../models').mkdir(parents=True, exist_ok=True)
joblib.dump(modelo_otimizado, '../models/modelo_random_forest.pkl')

print("Modelo salvo em models/modelo_random_forest.pkl")
